In [ ]:
# Created by: Sakhekile Jerry Mtyingizane
# Date created: 06/08/2026
# Objective: Writing the .csv datasets to the Bronze Delta tables


In [1]:
from pyspark.sql.functions import current_timestamp, lit

RAW_PATH = "Files/raw_data"

sources = {
    "olist_customers_dataset.csv":          "bronze_customers",
    "olist_geolocation_dataset.csv":        "bronze_geolocation",
    "olist_orders_dataset.csv":             "bronze_orders",
    "olist_order_items_dataset.csv":        "bronze_order_items",
    "olist_order_payments_dataset.csv":     "bronze_order_payments",
    "olist_order_reviews_dataset.csv":      "bronze_order_reviews",
    "olist_products_dataset.csv":           "bronze_products",
    "olist_sellers_dataset.csv":            "bronze_sellers",
    "product_category_name_translation.csv":"bronze_category_translation",
}

for file_name, table_name in sources.items():
    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .option("multiLine", "true")
          .option("escape", '"')
          .csv(f"{RAW_PATH}/{file_name}"))

    df = (df.withColumn("_ingested_at", current_timestamp())
            .withColumn("_source_file", lit(file_name)))

    df.write.format("delta").mode("overwrite").saveAsTable(table_name)
    print(f"{table_name:<32} {df.count():>8,} rows  {len(df.columns)} cols")

StatementMeta(, dfbe4e90-43e2-4e3b-a4ec-3ce97e58c1c5, 3, Finished, Available, Finished, False)

bronze_customers                   99,441 rows  7 cols
bronze_geolocation               1,000,163 rows  7 cols
bronze_orders                      99,441 rows  10 cols
bronze_order_items                112,650 rows  9 cols
bronze_order_payments             103,886 rows  7 cols
bronze_order_reviews               99,224 rows  9 cols
bronze_products                    32,951 rows  11 cols
bronze_sellers                      3,095 rows  6 cols
bronze_category_translation            71 rows  4 cols


## Bronze Reconciliation

Verifies that every Delta table in the Bronze layer contains the same number of
rows as the CSV it was loaded from.

**Why this exists.** The ingestion loop reports its own row counts, which is
circular, the same code that wrote the data is vouching for it. This cell re-reads
each source file independently and compares. Silent row loss during ingestion is
one of the hardest failures to detect downstream: a join simply returns fewer
results, and nothing raises an error.

**Parsing must match.** The raw read uses the same `multiLine` and `escape`
options as the ingest. The reviews file contains free-text customer comments with
embedded newlines and quote characters; parsed without these options, single
records split across multiple rows. Comparing differently-parsed reads would
produce mismatches unrelated to data integrity.

`inferSchema` is omitted here this cell counts rows rather than typing columns,
and inference requires an additional full pass over the data.

**Scope.** Row count detects wholesale loss or duplication. It does not detect
truncated values, incorrect types, or character corruption. Column-level profiling
belongs in the Silver layer, where types are enforced and business rules applied.

**Expected result:** PASS on all nine tables.


In [3]:
recon = []
for file_name, table_name in sources.items():
    raw_count = (spark.read
                 .option("header", "true")
                 .option("multiLine", "true")
                 .option("escape", '"')
                 .csv(f"{RAW_PATH}/{file_name}")
                 .count())
    delta_count = spark.table(table_name).count()
    recon.append((table_name, raw_count, delta_count, "PASS" if raw_count == delta_count else "FAIL"))

for name, raw, delta, status in recon:
    print(f"{status}  {name:<32} raw {raw:>9,}  delta {delta:>9,}")

StatementMeta(, dfbe4e90-43e2-4e3b-a4ec-3ce97e58c1c5, 5, Finished, Available, Finished, False)

PASS  bronze_customers                 raw    99,441  delta    99,441
PASS  bronze_geolocation               raw 1,000,163  delta 1,000,163
PASS  bronze_orders                    raw    99,441  delta    99,441
PASS  bronze_order_items               raw   112,650  delta   112,650
PASS  bronze_order_payments            raw   103,886  delta   103,886
PASS  bronze_order_reviews             raw    99,224  delta    99,224
PASS  bronze_products                  raw    32,951  delta    32,951
PASS  bronze_sellers                   raw     3,095  delta     3,095
PASS  bronze_category_translation      raw        71  delta        71
